# DashPVA Tools — Guided Tour

The utilities outside `DashAnalysis`: settings, file I/O, masking, and reciprocal-space conversion.

Set `FILENAME` below. Cells needing a real file or hardware report and continue rather than raising.

Companion: `DashAnalysis_Quickstart.ipynb` (slicing, line cuts, volumes).

**Setup.** Point `FILENAME` at one of your scans.

In [ ]:
import numpy as np

from dashpva.utils import HDF5Loader, MaskManager, RSMConverter
import dashpva.settings as app_settings

FILENAME = "your_data_file.h5"      # <- point this at a real scan

print("project root:", app_settings.PROJECT_ROOT)
print("output path :", app_settings.OUTPUT_PATH)

## 1. Settings and profiles

Every tool reads config from `dashpva.settings`, so viewers, consumers and notebooks agree.

**What's loaded.** `'db'` + profile id, `'toml'` + path, or `None` for defaults.

In [ ]:
print("source :", app_settings.SOURCE_TYPE)
print("locator:", app_settings.LOCATOR)
print("beamline:", app_settings.BEAMLINE_NAME or "(not set)")
print("output path:", app_settings.OUTPUT_PATH)

**Switch profile.** `set_locator()` then `reload()`. Assigning to the module directly gets overwritten.

In [ ]:
# Uncomment and adapt to switch profiles:
# app_settings.set_locator("profile:s6lambda1")
# app_settings.reload()

print("detector prefix:", app_settings.DETECTOR_PREFIX)
print("IOC prefix     :", app_settings.IOC_PREFIX)
print("input channel  :", app_settings.get_input_channel(""))
print("configured ROIs:", sorted(app_settings.ROI) or "(none)")

## 2. HDF5Loader — files in and out

Handles 2D stacks, 3D volumes and point clouds. Reports errors via `get_last_error()` instead of raising.

**Inspect first.** Lists every dataset without reading bulk data.

In [ ]:
loader = HDF5Loader()

if loader.validate_file(FILENAME):
    print(loader.get_file_info(FILENAME, style="text", summarize_datasets=True))
else:
    print("Not a readable HDF5 file:", loader.get_last_error())

**Load.** Returns `(points, intensities, n_frames, frame_shape)` — one row per pixel per frame.

In [ ]:
try:
    points, intensities, n_frames, frame_shape = loader.load_h5_to_3d(FILENAME)
    print(f"points      : {points.shape}")
    print(f"intensities : {intensities.shape}")
    print(f"frames      : {n_frames}  of shape {frame_shape}")
    print(f"HKL extent  : H {points[:,0].min():.3f}..{points[:,0].max():.3f}")
except Exception as e:
    print("Load failed:", e, "|", loader.get_last_error())

**Save.** Same layout the readers expect, so the Workbench can reopen it. Returns a bool — check it.

In [ ]:
# Example: keep only the brightest 1% and save that as a new file
# ok = loader.save_point_cloud_to_h5(
#     "bright_subset.h5",
#     points[intensities > np.percentile(intensities, 99)],
#     intensities[intensities > np.percentile(intensities, 99)],
#     metadata={"note": "top 1% by intensity"},
# )
# print("written:", ok)

## 3. MaskManager — bad pixels

One hot pixel can dominate a colour scale and skew every statistic.

**Detect.** Run `detect_hot_pixels` on **dark frames** or real Bragg peaks get masked. `detect_dead_pixels` needs illuminated frames.

In [ ]:
mask_mgr = MaskManager()

# Stand-in for a dark-frame stack: flat noise with three stuck pixels
dark = np.random.normal(100, 5, size=(20, 64, 64))
dark[:, 10, 10] = 5000
dark[:, 32, 40] = 4200
dark[:, 55, 3] = 6100

hot = mask_mgr.detect_hot_pixels(dark, sigma=5.0)
print("hot pixels found:", int(np.count_nonzero(hot)))
print("locations       :", list(zip(*np.nonzero(hot))))

**Combine and apply.** `combine_masks` ORs into the active mask. `apply_to_image` returns a copy — your array is untouched.

In [ ]:
mask_mgr.combine_masks(hot)

frame = dark[0]
cleaned = mask_mgr.apply_to_image(frame)

print("before:", frame.max(), " after:", cleaned.max())
print("original untouched:", frame.max() > cleaned.max())

**Look at it.** Raw, masked, and the mask itself.

In [ ]:
import matplotlib.pyplot as plt

# Stuck pixels are ~50x the background, so a shared linear scale would show
# three dots on black. Clip the display at the 99th percentile of the cleaned
# frame and ring the offenders -- otherwise raw and masked look identical and
# the whole story sits in the titles.
vmax = np.percentile(cleaned, 99)
ys, xs = np.nonzero(hot)
fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), constrained_layout=True)

for ax, img, title in (
    (axes[0], frame, f"raw  (max {frame.max():,.0f})"),
    (axes[1], cleaned, f"masked  (max {cleaned.max():,.0f})"),
):
    im = ax.imshow(img, cmap="magma", vmin=dark.min(), vmax=vmax, origin="lower")
    ax.set_title(title, fontsize=10)
    for x, y in zip(xs, ys):
        ax.add_patch(plt.Circle((x, y), 4, fill=False, color="#0072B2", lw=1.6))
fig.colorbar(im, ax=axes[:2], shrink=0.85, label="counts (clipped at p99)")

# The mask itself: binary state, so two flat tones rather than a ramp.
axes[2].imshow(hot, cmap="Greys", origin="lower", interpolation="nearest")
axes[2].set_title(f"mask  ({int(np.count_nonzero(hot))} px)", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

**Persist.** `save_active_mask` for reuse; `export_json_mask` in the form the detector IOC accepts.

In [ ]:
# path = mask_mgr.save_active_mask()          # -> masks/<timestamp>.npy
# mask_mgr.export_json_mask("mask_for_ioc.json", set_value=0)
# mask_mgr.load_mask(path, detector_shape=(64, 64))
mask_mgr.clear_mask()
print("mask cleared")

## 4. RSMConverter — angles to Q

Same conversion the live RSM consumer runs; use it here to check geometry offline.

**Geometry.** Read from the file's own metadata — a scan carries its geometry with it.

In [ ]:
import h5py

conv = RSMConverter()
try:
    with h5py.File(FILENAME, "r") as f:
        print("energy / UB :", conv.get_physics_params(f))
        sample, detector = conv.get_sample_and_detector_circles(f)
        print("sample circles  :", sample)
        print("detector circles:", detector)
except Exception as e:
    print("Geometry unavailable:", e)

**Convert all frames.** Returns flat `(N, 3)`. For big scans use `q_for_frames(...)` in chunks.

In [ ]:
try:
    q = conv.get_q_points(FILENAME)
    intens = conv.get_intensity(FILENAME)
    print("Q points  :", q.shape)
    print("intensity :", intens.shape)
    print("Qx range  :", f"{q[:,0].min():.4f} .. {q[:,0].max():.4f}")
except Exception as e:
    print("Conversion failed:", e)

**Look at it.** Where the scan actually sampled reciprocal space — gaps here
explain empty voxels later.

In [ ]:
import matplotlib.pyplot as plt

# Guarded: needs the conversion above to have produced Q.
if "q" in dir() and q is not None and len(q):
    fig, ax = plt.subplots(figsize=(5.5, 4.6), constrained_layout=True)
    # Millions of points -- hexbin, not scatter, which would be an opaque blob.
    hb = ax.hexbin(q[:, 0], q[:, 2], gridsize=90, bins="log", cmap="magma")
    ax.set_xlabel("Qx"); ax.set_ylabel("Qz")
    ax.set_title("measured coverage in Q (log density)", fontsize=10)
    ax.set_aspect("equal")
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    fig.colorbar(hb, ax=ax, label="points per cell (log)")
    plt.show()
else:
    print("No Q array in scope -- run the conversion cell against a real scan first.")

**One frame.** Quickest check that a geometry change did what you expected.

In [ ]:
try:
    qx, qy, qz = conv.create_rsm(FILENAME, frame=0)
    print("frame 0 ->", qx.shape, qy.shape, qz.shape)
except Exception as e:
    print("Single-frame conversion failed:", e)

## 5. Putting it together

Load → mask → convert → slice. Same calls the live viewers make.

In [ ]:
# from dashpva.utils import DashAnalysis
#
# points, intensities, n_frames, frame_shape = loader.load_h5_to_3d(FILENAME)
#
# keep = ~mask_mgr.mask.ravel() if mask_mgr.mask is not None else slice(None)
# points, intensities = points[keep], intensities[keep]
#
# da = DashAnalysis()
# sl = da.slice_data(data=(points, intensities), hkl='HL', shape=(512, 512), show=True)
# img, extent = da.show_slice(sl, return_image=True)
# cut = da.line_cut('zero', param=(0.0, 'x'), vol=(img, extent), width_px=3)

## Where each tool lives

| Tool | Module | Used by |
|---|---|---|
| `DashAnalysis` | `utils.dash_analysis` | offline slicing, cuts, volumes |
| `HDF5Loader` | `utils.hdf5_loader` | Workbench |
| `HDF5Writer` | `utils.hdf5_writer` | Scan Monitor, HKL 3D |
| `MaskManager` | `utils.mask_manager` | area-detector viewer |
| `RSMConverter` | `utils.rsm_converter` | RSM consumers |
| `PVAReader` | `utils.pva_reader` | every live viewer |
| settings | `dashpva.settings` | everything |

`PVAReader` and `HDF5Writer` need a live channel, so they belong in the viewers.